In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
sys.path.insert(0, str(EXT / "diffmot"))               # agar from external.* resolve
sys.path.insert(0, str(EXT / "diffmot" / "external"))  # agar import fast_reid / YOLOX resolve (fast_reid tidak punya setup.py)
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

# 40 — DiffMOT: Patch Eval + Smoke ReID

**Kernel: `s2-diffmot`** (python 3.9 + torch 2.0.1 cu118). Pastikan kernel sudah dipilih.

Alasan patch: di `diffmot.py` eval, `cv2.imread` dikomentari → `compute_embedding(img=None,...)`
crash bila cache `{reid_dir}/{seq}_embedding.pkl` belum ada. Patch 2 baris: baca img dan
teruskan ke `tracker.update(...)` — cache terisi otomatis saat run pertama, reuse setelahnya.
`scripts/s2/patch_diffmot_eval.py` idempotent.

Smoke test: load bobot ReID (FastReID) + forward dummy — validasi CUDA/weight SEBELUM run penuh.

In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

In [ ]:
# sys.path untuk import diffmot
import sys
sys.path.insert(0, str(EXT / "diffmot"))
print(sys.path[:2])

In [ ]:
!python $S2_ROOT/scripts/s2/patch_diffmot_eval.py --diffmot-root $S2_EXT/diffmot

In [ ]:
# verifikasi baris hasil patch
src = (EXT / "diffmot" / "diffmot.py").read_text()
import re
for pat in ["img = cv2.imread(im_path)", "tag, img)"]:
    print(pat, "->", "OK" if pat in src else "MISSING")

### Smoke ReID model (FastReID)

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# load bobot ReID DanceTrack — validasi path & arsitektur
from external.adaptors.fastreid_adaptor import FastReID
w_dance = EXT / "diffmot" / "external" / "weights" / "dance_sbs_S50.pth"
assert w_dance.exists(), f"tidak ada {w_dance} — jalankan notebook 10 dulu"
m = FastReID(str(w_dance))
m.eval(); m.cuda(); m.half()
x = torch.randn(1, 3, 384, 128).half().cuda()   # (N,C,H,W) sesuai crop_size (128,384) di embedding.py
with torch.no_grad():
    y = m(x)
print("ReID forward OK, output:", tuple(y.shape))

### Smoke pipeline embedding + cache (1 frame, 1 sekuens)

In [ ]:
import cv2, numpy as np
from tracker.embedding import EmbeddingComputer

class Cfg:
    reid_dir = str(DATA / "embeddings" / "dance")

ec = EmbeddingComputer(Cfg(), "dance", True, True)   # dataset='dance' -> pakai dance_sbs_S50.pth
seq0 = sorted(p for p in (DATA / "dancetrack" / "val").iterdir() if p.is_dir())[0]
img = cv2.imread(str(sorted((seq0 / "img1").glob("*.*"))[0]))
bbox = np.array([[10, 10, 120, 240], [200, 60, 320, 300]], np.float32)
emb = ec.compute_embedding(img, bbox, f"{seq0.name}:1")
print("embedding shape:", emb.shape)
ec.dump_cache()
print("cache:", Cfg.reid_dir)

**Lanjut**: `50_s2_run_diffmot.ipynb` (kernel `s2-diffmot`).